# S4 : E1, XGBoost avec les statistiques du graphe causal

E1 ajoute au scenario B (type, montant) des variables calculees sur l'historique des comptes, avec une seule regle : pour une transaction au step t, on n'utilise que les transactions des steps strictement avant t, tous types confondus, sans aucune etiquette.

On suit la fiche protocole version 2 et le registre : memes partitions, memes reglages possibles que E0, reglages choisis sur la validation, test evalue une fois.

In [ ]:
!git clone -q https://github.com/kenzaez/pfa06-graph-fraud.git
%cd pfa06-graph-fraud
!pip install -q -r requirements.txt

## 1. Donnees

In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

In [ ]:
!kaggle datasets download -d ealaxi/paysim1 -p data
!unzip -o -q data/paysim1.zip -d data

In [ ]:
from src.audit import load, sha256

PATH = "data/PS_20174392719_1491204439457_log.csv"
assert sha256(PATH) == "16910f90577b0d981bf8ff289714510bb89bc71bff7d3f220f024e287e4eea6b"
df = load(PATH)
df.shape

## 2. Tests de causalite

Avant tout entrainement. Un evenement futur ne change rien au passe, melanger les lignes ne change rien, les etiquettes ne sont jamais lues, un compte nouveau n'a pas d'historique, et une petite trajectoire est verifiee a la main.

In [ ]:
!python -m pytest -q tests/test_causalite.py tests/test_protocol.py tests/test_features.py

## 3. Variables du graphe sur les vraies donnees

In [ ]:
from src import graphe_causal
from src.features import prepare

d = graphe_causal.ajouter(prepare(df))
d[graphe_causal.NOMS].describe().T.round(3)

Sur une partie des donnees, on refait le calcul en coupant l'historique juste avant un step choisi : les variables des transactions avant ce step doivent etre identiques. C'est le test de fuite sur les vraies donnees.

In [ ]:
import numpy as np

T = 500
coupe = df[df.step <= T]
g_coupe = graphe_causal.caracteristiques(coupe)
g_plein = d.loc[coupe.index, graphe_causal.NOMS]
np.nanmax(np.abs(g_coupe.values - g_plein.values))

## 4. Part de transactions avec historique

In [ ]:
from src.features import population

p = population(d, "population_principale")
p.groupby("partition")[["g_orig_a_historique", "g_dest_a_historique"]].mean().round(4)

Dans PaySim presque tous les emetteurs sont nouveaux (99,84 %). Si la part avec historique est tres faible, E1 a peu d'information relationnelle a exploiter. C'est un resultat a rapporter tel quel.

## 5. E0 scenario B de reference, refait avec les reglages deja choisis

Meme code, memes reglages que `config_choisie.json`, pour avoir les scores de E0 B transaction par transaction et comparer sur les memes lignes.

In [ ]:
import json
from pathlib import Path
from src import e0

dossier = Path("reports/e0")
dossier.mkdir(parents=True, exist_ok=True)
Path("models").mkdir(exist_ok=True)
choix = json.loads(Path("reports/e0/config_choisie.json").read_text())
r_b = e0.evaluer(prepare(df), "B", "population_principale", params=choix["B"])
r_b["metriques"][(r_b["metriques"].partition == "test")][["modele", "ap", "ratio_ap_taux"]].round(4)

## 6. Entrainement de E1

In [ ]:
from src import e1

tab, params = e1.main(df)
params

In [ ]:
tab[(tab.population == "population_principale")][["modele", "partition", "ap", "ratio_ap_taux", "P@250", "R@250"]].round(4)

## 7. Comparaison E0 B contre E1 sur les memes transactions

In [ ]:
from src import comparaison_e0_e1 as cmp

res = cmp.main(d)
res["global"].round(4)

In [ ]:
res["par_jour"].round(4)

In [ ]:
res["historique"].round(4)

## 8. Ce qu'il faut recuperer

Telecharger `reports/e1`, `reports/comparaison`, `models/e1` et le fichier `reports/e0/predictions_B_population_principale.csv.gz`, puis les mettre dans le depot.

In [ ]:
!zip -q -r sorties_s4.zip reports/e1 reports/comparaison models/e1 reports/e0/predictions_B_population_principale.csv.gz
from google.colab import files
files.download("sorties_s4.zip")